In [1]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)

%pwd


'/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model'

In [ ]:


import pandas as pd 
import geopandas as gpd
import numpy as np
import requests
import seaborn as sns
import matplotlib.pyplot as plt

from shapely.geometry import LineString, Point
from shapely.affinity import translate

import traffic.utils.analysis_utils as au



# Make api request

In [ ]:
# Define the base URL and parameters
base_url = "https://maps.udot.utah.gov/central/rest/services/TrafficAndSafety/UDOT_Speed_Limits/MapServer/0/query"
params = {
    "where": "Name='0210'",
    "outFields": "*",
    "f": "json"
}

# Make the GET request
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    data  = response.json()
    features = data['features']
    # Process the data as needed
else:
    print(f"Error: {response.status_code}")

# Convert to a gdf of points

In [ ]:
geoms = []
speed_limits = []
# this loops through the api request and gets the paths and speed limits
for feature in features:
    paths = feature["geometry"]["paths"]
    for path in paths:
        geoms.append(LineString(path))
        speed_limits.append(feature["attributes"]["Speed_Limit"])

# saves the data in a gdf 
gdf = gpd.GeoDataFrame({"speed_limit": speed_limits}, geometry=geoms, crs="EPSG:26912").to_crs(epsg=32612)

# i drop the snowbird loop
gdf = gdf.drop(1).reset_index(drop=True) 

# reorder the rows so that the points are in order
gdf = gdf.reindex([3, 2, 1, 0]).reset_index(drop=True)



In [ ]:
def densify_lines_to_points(gdf_lines, spacing_meters=5):
    '''
    Convert LineStrings into evenly spaced Points with inherited attributes.

    Parameters:
        gdf_lines: GeoDataFrame with LineStrings and any attributes (e.g., speed_limit)
        spacing_meters: float, distance between points (in meters)

    Returns:
        GeoDataFrame of Points with inherited attributes
    '''
    points_data = []

    for idx, row in gdf_lines.iterrows():
        line = row.geometry
        length = line.length
        num_points = max(int(length // spacing_meters), 1)
        distances = [i * spacing_meters for i in range(num_points + 1)]
        distances = [d if d <= length else length for d in distances]  # cap at end
        
        for d in distances:
            point = line.interpolate(d)
            data = row.drop('geometry').to_dict()  # copy all other attributes
            data['geometry'] = point
            points_data.append(data)

    return gpd.GeoDataFrame(points_data, crs=gdf_lines.crs)


points_gdf = densify_lines_to_points(gdf, spacing_meters=50)
points_gdf.head(2)

# Make curvature cols

Curvature as i have defined it represents the change in degrees per 100m. Ie a standardized delta heading. 

How frame shifts affected the calculations
1) meters_to_next describes the distance(m) from index = i to index=i+1, the value is recorded on the ith row
2) heading is the direction (degrees, due east = 0) of the vector created by the ith and ith +1 point, it is recorded on the ith row
3) delta_heading is the abs(ith heading - ith +1 heading), this involves data from points i, i+1 and i+2
4) curvature is ith delta_heading/ith meters_to_next

There is a potential that calculating things this way is illogical. Maybe the info should be recorded on the row representing the max(i) point involved in the calculation
Also standardizing curvature as 100*delta_heading/(ith meters_to_next) might need to be changed to 100*delta_heading/sum(ith meters_to_next, ith+1 meters_to_next)

In [ ]:
road_gdf = points_gdf.copy()

In [ ]:
road_gdf['meters_to_next'] = road_gdf.geometry.distance(road_gdf.geometry.shift(-1)).fillna(0) # units of meters

def calculate_heading(point1, point2):
    if point1 is None or point2 is None:
        return np.nan
    dx = point2.x - point1.x
    dy = point2.y - point1.y
    return np.degrees(np.arctan2(dy, dx))

# the actual calculation of heading
road_gdf['heading'] = road_gdf.geometry.combine(road_gdf.geometry.shift(-1), calculate_heading)

# just a bunch of manipulation of the heading col 
road_gdf['delta_heading'] = (road_gdf['heading'].shift(-1) - road_gdf['heading']).fillna(0)
road_gdf['curvature'] = round(((road_gdf['delta_heading'].abs() / road_gdf['meters_to_next']) * 100),3).fillna(0) # i clipped the curvature at 90 degrees
road_gdf['radius_ft'] = 59055.12 / (np.pi * road_gdf['curvature'])

# calculated radius_ft because it lines up with the reccomended curve speed limits in the following dot document
# https://highways.dot.gov/safety/rwd/keep-vehicles-road/horizontal-curve/low-cost-treatments-horizontal-curve-safety-2016-4
road_gdf['linked_coord'] = None
road_gdf

In [ ]:
#sns.lineplot(data=road_gdf, x=road_gdf.index, y='curvature')
#sns.histplot(road_gdf.curvature)
#sns.ecdfplot(road_gdf.curvature)

# Adding road segment 

In [ ]:
# establishing the index range based on speed_limit from the current gdf
gdf_reset = road_gdf.reset_index()
gdf_reset.groupby("speed_limit").agg(min_index=("index", "min"), max_index=("index", "max"))

In [ ]:
# Build conditions and labels
conditions = [
    gdf_reset["index"].between(0, 118),
    gdf_reset["index"].between(119, 218),# this splits up the 40 mph into two segments it was just too big 
    gdf_reset["index"].between(219, 318),# p2 two of the split
    gdf_reset["index"].between(319, 369), 
    gdf_reset["index"].between(370, 401),
]
labels = [1, 2, 3, 4, 5]

# Assign the new column
road_gdf["road_section"] = np.select(conditions, labels, default=0)

au.plot_colored_road(road_gdf, 'road_section')
road_gdf.head(2)


# Manually adjusting the speed limit at the start of the road

In [ ]:
road_gdf.loc[0:2, 'speed_limit'] = [10,20,35]
road_gdf

# Add distance traveled (meters)

In [ ]:
road_gdf['distance_traveled'] = road_gdf['meters_to_next'].shift(fill_value=0).cumsum()
road_gdf.tail(2)

# Create down lane (not used)

In [ ]:
# reverse the gdf 
reverse_road_gdf = road_gdf[::-1]
# shift the road over
reverse_road_gdf["geometry"] = reverse_road_gdf["geometry"].apply(lambda geom: translate(geom, xoff=300, yoff=500))
# concat them together
full_road = pd.concat([road_gdf, reverse_road_gdf]).reset_index(drop=True)

# adding a col that links the up and down lanes 
full_road['linked_coord'] = full_road.geometry[::-1 ].reset_index(drop=True)

# Outputs

In [ ]:
road_gdf.head()

In [ ]:


# Save to the correct data/roads/ folder from other_notebooks/
road_gdf.to_parquet("../data/roads/hw210_sl_and_curvs.parquet", index=True)
road_gdf.head()


In [ ]:
full_road.to_parquet("data/roads/hw210_full_road.parquet", index=False)

full_road.head()